# API metrics: issues, comments, reviews and response times

Git only contains merged work. This notebook uses the GitHub REST API to add what git cannot see: PRs closed without merging, issues, comments, reviews and how fast people respond. It completes the *Current contributors and practices* subsection (activity metrics and process metrics).

**How to run:** put `GITHUB_TOKEN=...` in a `.env` file next to this notebook (a classic token without any scopes is enough; keep `.env` in `.gitignore`) and run all cells. Collection takes about 1-2 hours because of GitHub's rate limit (5000 requests per hour). Every page is saved in `api_cache/`, so if the notebook stops, just run it again and it continues where it left off.

It needs the files written by notebooks 1 and 3 in `data/`.

Author groups are the same as in the rest of the report, based on merged contributions in git:
- **Frequent**: frequent contributor in 2025 (notebook 3)
- **Occasional**: had merged work before 2025, but not frequent (occasional, returning and dormant people)
- **Newcomer**: no merged contribution before 2025, including people whose PRs were never merged

In [1]:
import json, os, re, time
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

In [2]:
# constants

# the token is read from the .env file next to this notebook (never paste it here)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("python-dotenv not installed: pip install python-dotenv")
token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError("no GITHUB_TOKEN found: add GITHUB_TOKEN=... to your .env file")
api = "https://api.github.com"
owner = repo = "kubernetes"
start = pd.Timestamp("2025-01-01", tz="UTC")
end = pd.Timestamp("2026-01-01", tz="UTC")
cache = Path("api_cache")
cache.mkdir(exist_ok=True)

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"})

In [3]:
# check the token and the rate limit

r = session.get(f"{api}/rate_limit", timeout=30)
if r.status_code == 401:
    raise RuntimeError("token not accepted, check the token in the constants cell")
core = r.json()["resources"]["core"]
print("requests left this hour:", core["remaining"], "of", core["limit"])

requests left this hour: 4991 of 5000


In [4]:
# request one page, waiting when GitHub's rate limit is reached

class PaginationLimit(Exception):
    pass

def request(url):
    for attempt in range(10):
        try:
            r = session.get(url, timeout=60)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            print(f"  no answer from GitHub ({type(e).__name__}), retry {attempt + 1}/10", flush=True)
            time.sleep(10 * (attempt + 1))
            continue
        if r.status_code == 200:
            return r
        if r.status_code == 422 and "pagination" in r.text.lower():
            raise PaginationLimit()
        if r.status_code in (403, 429):
            if r.headers.get("X-RateLimit-Remaining") == "0":
                wait = int(r.headers.get("X-RateLimit-Reset", time.time() + 60)) - time.time() + 5
            else:
                wait = int(r.headers.get("Retry-After", 60))
            print(f"rate limit reached, waiting {max(wait, 5) / 60:.1f} min", flush=True)
            time.sleep(max(wait, 5))
            continue
        if r.status_code >= 500:
            print(f"  GitHub error {r.status_code}, retry {attempt + 1}/10", flush=True)
            time.sleep(10 * (attempt + 1))
            continue
        raise RuntimeError(f"GitHub said {r.status_code}: {r.text[:200]}")
    raise RuntimeError("giving up on " + url)

# follow all pages of a list, keep only the fields we need, save every page in the cache
# GitHub stops long lists after 300 pages; then we continue with the comments
# *updated* since the newest date seen so far, sorted by update time (fast for GitHub)

def fetch_all(name, url, params, keep, stop=None, restart=False):
    base = url
    url = requests.Request("GET", base, params=params).prepare().url
    rows, page, last_since = [], 0, None
    seg_rows, seg_key = [], "updated_at" if params.get("sort") == "updated" else "created_at"
    while url:
        f = cache / f"{name}_{page:05d}.json"
        if f.exists():
            saved = json.loads(f.read_text())
        else:
            try:
                r = request(url)
            except PaginationLimit:
                # resume from the newest date of the key this part of the list was sorted by
                since = max(x.get(seg_key) or x["created_at"] for x in seg_rows)
                print(f"{name}: GitHub limits deep pages, continuing with comments updated since {since}", flush=True)
                url = requests.Request("GET", base, params={**params, "sort": "updated", "since": since}).prepare().url
                seg_rows, seg_key = [], "updated_at"
                continue
            saved = {"rows": [keep(x) for x in r.json()], "next": r.links.get("next", {}).get("url")}
            f.write_text(json.dumps(saved))
        rows += saved["rows"]
        seg_rows += saved["rows"]
        url = saved["next"]
        page += 1
        if page % 25 == 0:
            print(f"{name}: {page} pages, {len(rows)} rows", flush=True)
        if stop and saved["rows"] and stop(saved["rows"]):
            break
        # GitHub silently stops after 300 pages (30,000 items) without an error:
        # if the list ends on a full page, continue from the newest date seen so far
        if url is None and restart and len(saved["rows"]) == 100:
            # resume from the newest date of the key this part of the list was sorted by
            since = max(x.get(seg_key) or x["created_at"] for x in seg_rows)
            if since != last_since:
                last_since = since
                print(f"{name}: GitHub stopped after {page} pages, continuing with comments updated since {since}", flush=True)
                url = requests.Request("GET", base, params={**params, "sort": "updated", "since": since}).prepare().url
                seg_rows, seg_key = [], "updated_at"
    print(f"{name}: done, {page} pages, {len(rows)} rows, newest {max(x['created_at'] for x in rows) if rows else '-'}")
    return pd.DataFrame(rows)

## Collect the data

In [5]:
# 1. all issues and PRs created in 2025 (newest first, stop once we pass 1 Jan 2025)

def keep_item(x):
    user = x.get("user") or {"login": "ghost", "type": "User"}
    return {"number": x["number"], "is_pr": "pull_request" in x,
            "author": user["login"], "author_type": user["type"],
            "created_at": x["created_at"], "closed_at": x["closed_at"],
            "merged_at": (x.get("pull_request") or {}).get("merged_at")}

items = fetch_all("items", f"{api}/repos/{owner}/{repo}/issues",
                  {"state": "all", "sort": "created", "direction": "desc", "per_page": 100},
                  keep_item, stop=lambda rows: rows[-1]["created_at"] < start.isoformat())
for col in ["created_at", "closed_at", "merged_at"]:
    items[col] = pd.to_datetime(items[col], utc=True)
items = items[(items.created_at >= start) & (items.created_at < end)].drop_duplicates("number")
print(len(items), "items created in 2025 (bots not yet removed):", items.is_pr.sum(), "PRs,", (~items.is_pr).sum(), "issues")

items: 25 pages, 2500 rows
items: 50 pages, 5000 rows
items: 75 pages, 7500 rows
items: 100 pages, 10000 rows
items: 125 pages, 12500 rows
items: done, 135 pages, 13500 rows, newest 2026-10-18T13:16:52Z
6428 items created in 2025 (bots not yet removed): 4569 PRs, 1859 issues


In [6]:
# 2. all comments since 1 Jan 2025 (also 2026, so that late responses to 2025 PRs are seen)
# Prow commands like /lgtm are at the start of a line; bot messages are long, so we check the full text here

def flags(body):
    body = body or ""
    return {"lgtm": bool(re.search(r"(?m)^\s*/lgtm\b", body)),
            "approve": bool(re.search(r"(?m)^\s*/approve\b", body)),
            "ok_to_test": bool(re.search(r"(?m)^\s*/ok-to-test\b", body)),
            "needs_ok_to_test": "verify that this patch is reasonable to test" in body,
            "stale": "/lifecycle stale" in body,
            "rotten": "/lifecycle rotten" in body,
            "close": bool(re.search(r"(?m)^\s*/close\b", body)),
            "excerpt": body[:200]}

def keep_comment(x):
    user = x.get("user") or {"login": "ghost", "type": "User"}
    return {"id": x["id"], "number": int(x["issue_url"].rsplit("/", 1)[1]), "login": user["login"],
            "user_type": user["type"], "created_at": x["created_at"], "updated_at": x.get("updated_at"),
            "kind": "comment", **flags(x["body"])}

comments = fetch_all("comments", f"{api}/repos/{owner}/{repo}/issues/comments",
                     {"sort": "created", "direction": "asc", "since": start.isoformat(), "per_page": 100},
                     keep_comment, restart=True)

comments: 25 pages, 2500 rows
comments: 50 pages, 5000 rows
comments: 75 pages, 7500 rows
comments: 100 pages, 10000 rows
comments: 125 pages, 12500 rows
comments: 150 pages, 15000 rows
comments: 175 pages, 17500 rows
comments: 200 pages, 20000 rows
comments: 225 pages, 22500 rows
comments: 250 pages, 25000 rows
comments: 275 pages, 27500 rows
comments: 300 pages, 30000 rows
comments: GitHub stopped after 300 pages, continuing with comments updated since 2025-05-28T17:54:34Z
comments: 325 pages, 32500 rows
comments: 350 pages, 35000 rows
comments: 375 pages, 37500 rows
comments: 400 pages, 40000 rows
comments: 425 pages, 42500 rows
comments: 450 pages, 45000 rows
comments: 475 pages, 47500 rows
comments: 500 pages, 50000 rows
comments: 525 pages, 52500 rows
comments: 550 pages, 55000 rows
comments: 575 pages, 57500 rows
comments: 600 pages, 60000 rows
comments: GitHub stopped after 600 pages, continuing with comments updated since 2025-10-15T17:03:30Z
comments: 625 pages, 62500 rows
co

In [7]:
# 3. all inline review comments (comments on lines of code) since 1 Jan 2025

def keep_review_comment(x):
    user = x.get("user") or {"login": "ghost", "type": "User"}
    return {"id": x["id"], "number": int(x["pull_request_url"].rsplit("/", 1)[1]), "login": user["login"],
            "user_type": user["type"], "created_at": x["created_at"], "updated_at": x.get("updated_at"),
            "kind": "review_comment", **flags(x["body"])}

review_comments = fetch_all("review_comments", f"{api}/repos/{owner}/{repo}/pulls/comments",
                            {"sort": "created", "direction": "asc", "since": start.isoformat(), "per_page": 100},
                            keep_review_comment, restart=True)

com = pd.concat([comments.drop_duplicates("id"), review_comments.drop_duplicates("id")], ignore_index=True)
com["created_at"] = pd.to_datetime(com.created_at, utc=True)
com = com[com.created_at >= start]
items.to_csv("data/api_items_2025.csv", index=False)
com.drop(columns=["excerpt"]).to_csv("data/api_comments.csv", index=False)
print(len(com), "comments since 1 Jan 2025")

review_comments: 25 pages, 2500 rows
  no answer from GitHub (ConnectionError), retry 1/10
  GitHub error 500, retry 2/10
  GitHub error 502, retry 1/10
  GitHub error 502, retry 1/10
review_comments: 50 pages, 5000 rows
review_comments: 75 pages, 7500 rows
review_comments: 100 pages, 10000 rows
review_comments: 125 pages, 12500 rows
review_comments: 150 pages, 15000 rows
review_comments: 175 pages, 17500 rows
review_comments: 200 pages, 20000 rows
review_comments: 225 pages, 22500 rows
review_comments: 250 pages, 25000 rows
review_comments: 275 pages, 27500 rows
review_comments: 300 pages, 30000 rows
review_comments: 325 pages, 32500 rows
review_comments: 350 pages, 35000 rows
review_comments: 375 pages, 37500 rows
review_comments: 400 pages, 40000 rows
review_comments: 425 pages, 42500 rows
review_comments: 450 pages, 45000 rows
review_comments: 475 pages, 47500 rows
review_comments: 500 pages, 50000 rows
review_comments: 525 pages, 52500 rows
review_comments: 550 pages, 55000 rows
r

## Filtering: bots and author groups

Bots are detected with the same name measure as in notebook 1, plus GitHub's own account type `Bot`. Logins are mapped to people with the identity merging of notebook 1.

In [8]:
person_map = pd.read_csv("data/person_map.csv")
pid = dict(zip(person_map.node, person_map.person_id))
bot_logins = {n[6:] for n in person_map[person_map.is_bot].node if n.startswith("login:")}
bot_re = re.compile(r"(\[bot\]|-bot\b|-robot\b|\brobot\b|\bbot\b)", re.I)

def is_bot(login, user_type="User"):
    return user_type == "Bot" or bool(bot_re.search(login)) or login.lower() in bot_logins

com["is_bot"] = [is_bot(l, t) for l, t in zip(com.login, com.user_type)]
items = items[[not is_bot(a, t) for a, t in zip(items.author, items.author_type)]].copy()

c25 = com[com.created_at < end]
print(f"bot share of all comments in 2025: {c25.is_bot.mean():.1%}")
c25[c25.is_bot].login.value_counts().head(5)

bot share of all comments in 2025: 26.7%


login
k8s-ci-robot                     21102
k8s-triage-robot                  6272
linux-foundation-easycla[bot]      433
Copilot                             60
k8s-publishing-bot                  33
Name: count, dtype: int64

In [9]:
categories = pd.read_csv("data/people_2025.csv", index_col=0).category
first = pd.to_datetime(pd.read_csv("data/first_contribution.csv", index_col=0).first_contribution, utc=True)

def group(login):
    p = pid.get("login:" + login.lower())
    if p is None or p < 0 or p not in first.index or first[p] >= start:
        return "Newcomer"                 # no merged contribution before 2025
    if categories.get(p) == "Frequent":
        return "Frequent"
    return "Occasional"                   # earlier merged work, not frequent in 2025

items["group"] = items.author.map(group)
order = ["Frequent", "Occasional", "Newcomer"]
pd.crosstab(items.group, items.is_pr.map({True: "PRs", False: "issues"})).reindex(order, fill_value=0)

is_pr,PRs,issues
group,,
Frequent,2549,681
Occasional,556,256
Newcomer,1462,922


## Activity metrics (all activity types)

Per month in 2025: PRs and issues opened, comments by humans, inline review comments, and the review commands `/lgtm` (a reviewer accepts the change) and `/approve` (an approver accepts it).

In [10]:
h25 = c25[~c25.is_bot]
monthly = pd.DataFrame({
    "prs_opened": items[items.is_pr].groupby(items.created_at.dt.month).size(),
    "issues_opened": items[~items.is_pr].groupby(items.created_at.dt.month).size(),
    "human_comments": h25[h25.kind == "comment"].groupby(h25.created_at.dt.month).size(),
    "review_comments": h25[h25.kind == "review_comment"].groupby(h25.created_at.dt.month).size(),
    "lgtm": h25[h25.lgtm].groupby(h25.created_at.dt.month).size(),
    "approve": h25[h25.approve].groupby(h25.created_at.dt.month).size(),
    "commenters": h25.groupby(h25.created_at.dt.month).login.nunique()}).fillna(0).astype(int)
monthly.loc["total"] = monthly.sum()
monthly.loc["total", "commenters"] = h25.login.nunique()
monthly

,prs_opened,issues_opened,human_comments,review_comments,lgtm,approve,commenters
created_at,,,,,,,
1,324,163,3652,1776,278,185,486
2,401,151,4213,2674,331,211,480
3,415,194,5735,4181,476,298,551
4,290,147,2928,1235,169,128,457
5,333,140,3505,1841,267,171,472
6,407,177,3626,1781,282,197,448
7,519,181,5637,4733,465,340,526
8,282,153,3266,1057,157,98,493
9,393,135,3959,1763,300,214,448


In [11]:
# how concentrated is reviewing? (people needed for half of all /lgtm and /approve commands)

def people_for_half(logins):
    s = logins.str.lower().value_counts()
    return int((s.cumsum() / s.sum() < 0.5).sum() + 1), len(s)

lgtm_half, lgtm_people = people_for_half(h25[h25.lgtm].login)
approve_half, approve_people = people_for_half(h25[h25.approve].login)
print(f"/lgtm:    {lgtm_people} people, {lgtm_half} of them give half of all")
print(f"/approve: {approve_people} people, {approve_half} of them give half of all")

/lgtm:    176 people, 11 of them give half of all
/approve: 95 people, 4 of them give half of all


## Process metrics

### PR outcome by author group
Acceptance rate = merged / (merged + closed without merging), for PRs opened in 2025 (status at the time of collection).

In [12]:
prs = items[items.is_pr].copy()
prs["outcome"] = "open"
prs.loc[prs.closed_at.notna(), "outcome"] = "closed unmerged"
prs.loc[prs.merged_at.notna(), "outcome"] = "merged"

outcome = pd.crosstab(prs.group, prs.outcome).reindex(index=order, columns=["merged", "closed unmerged", "open"], fill_value=0)
outcome["acceptance"] = (outcome["merged"] / (outcome["merged"] + outcome["closed unmerged"])).round(3)
outcome

outcome,merged,closed unmerged,open,acceptance
group,,,,
Frequent,2018,515,16,0.797
Occasional,345,203,8,0.630
Newcomer,648,789,25,0.451


### Time to first human response (CHAOSS)
Days from opening until the first comment by a human who is not the author.

In [13]:
human = com[~com.is_bot].merge(items[["number", "author", "created_at"]], on="number", suffixes=("", "_item"))
human = human[(human.login.str.lower() != human.author.str.lower()) & (human.created_at >= human.created_at_item)]
first_response = human.groupby("number").created_at.min()

items["first_response"] = pd.to_datetime(items.number.map(first_response), utc=True)
items["ttfr_days"] = (items.first_response - items.created_at).dt.total_seconds() / 86400

def summary(df):
    return pd.Series({"items": len(df), "median_days": round(df.ttfr_days.median(), 2),
                      "p75_days": round(df.ttfr_days.quantile(0.75), 2),
                      "no_human_response": round(df.first_response.isna().mean(), 3)})

ttfr_pr = items[items.is_pr].groupby("group").apply(summary).reindex(order)
ttfr_issue = items[~items.is_pr].groupby("group").apply(summary).reindex(order)
pd.concat({"PRs": ttfr_pr, "issues": ttfr_issue})

items  median_days  p75_days  no_human_response
       group                                                       
PRs    Frequent    2549.0         0.34      1.81              0.113
       Occasional   556.0         0.37      2.91              0.112
       Newcomer    1462.0         0.57      2.91              0.172
issues Frequent     681.0         0.42      2.98              0.186
       Occasional   256.0         0.45      2.68              0.074
       Newcomer     922.0         0.43      2.09              0.118

### Bots and waiting PRs
How often does the lifecycle bot (`k8s-triage-robot`) mark a PR stale or rotten before any human has responded, and how many unmerged PRs does it close?

In [14]:
triage = com[com.login == "k8s-triage-robot"]
first_stale = triage[triage.stale | triage.rotten].groupby("number").created_at.min()
prs = items[items.is_pr].copy()
prs["stale_at"] = pd.to_datetime(prs.number.map(first_stale), utc=True)
prs["stale_before_response"] = prs.stale_at.notna() & (prs.first_response.isna() | (prs.stale_at < prs.first_response))

last_close = triage[triage.close].groupby("number").created_at.max()
prs["close_cmd"] = pd.to_datetime(prs.number.map(last_close), utc=True)
unmerged = prs[prs.closed_at.notna() & prs.merged_at.isna()].copy()
unmerged["closed_by_bot"] = (unmerged.closed_at - unmerged.close_cmd).abs() < pd.Timedelta(hours=1)

bots_table = pd.DataFrame({
    "PRs": prs.groupby("group").size(),
    "marked stale before any human response": prs.groupby("group").stale_before_response.mean().round(3),
    "closed unmerged": unmerged.groupby("group").size(),
    "of which closed by the bot": unmerged.groupby("group").closed_by_bot.mean().round(3)}).reindex(order)
bots_table

,PRs,marked stale before any human response,closed unmerged,of which closed by the bot
group,,,,
Frequent,2549,0.011,515,0.212
Occasional,556,0.034,203,0.384
Newcomer,1462,0.050,789,0.407


### Waiting for `/ok-to-test`
PRs from people outside the Kubernetes organisation only get CI after a member writes `/ok-to-test`. Prow announces this with a comment; we measure how long the PR then waits.

In [15]:
ci = com[com.login == "k8s-ci-robot"]
gated_at = ci[ci.needs_ok_to_test].groupby("number").created_at.min()
ok_at = com[com.ok_to_test & ~com.is_bot].groupby("number").created_at.min()

prs["gated"] = prs.number.isin(gated_at.index)
prs["ok_wait_days"] = (pd.to_datetime(prs.number.map(ok_at), utc=True) - prs.created_at).dt.total_seconds() / 86400
g = prs[prs.gated]
ok_table = pd.DataFrame({
    "share of PRs gated": prs.groupby("group").gated.mean().round(3),
    "gated PRs": g.groupby("group").size(),
    "got /ok-to-test": g.groupby("group").ok_wait_days.apply(lambda s: round(s.notna().mean(), 3)),
    "median wait (days)": g.groupby("group").ok_wait_days.median().round(2)}).reindex(order)
ok_table

,share of PRs gated,gated PRs,got /ok-to-test,median wait (days)
group,,,,
Frequent,0.043,110,0.800,0.46
Occasional,0.300,167,0.725,0.90
Newcomer,0.852,1246,0.599,0.84


### Do the approvers review?
Share of the approvers listed in the `OWNERS` files (notebook 3) who wrote at least one `/lgtm`, `/approve` or review comment in 2025.

In [16]:
approvers = set(pd.read_csv("data/approvers.csv").login.str.lower())
reviewing = h25[h25.lgtm | h25.approve | (h25.kind == "review_comment")]
active_reviewers = set(reviewing.login.str.lower())
per_approver = reviewing.assign(l=reviewing.login.str.lower())
per_approver = per_approver[per_approver.l.isin(approvers)].groupby("l").size().sort_values(ascending=False)

print("approvers:", len(approvers))
print("reviewed in 2025:", len(approvers & active_reviewers), f"({len(approvers & active_reviewers) / len(approvers):.0%})")
print(f"top 10 approvers do {per_approver.head(10).sum() / per_approver.sum():.0%} of all approver review actions")
per_approver.head(10)

approvers: 163
reviewed in 2025: 110 (67%)
top 10 approvers do 56% of all approver review actions


l
liggitt        3133
pohly          2523
macsko         1704
aojea           987
tallclair       881
thockin         830
dims            626
serathius       585
bentheelder     580
yliaog          568
dtype: int64

## Values for the report

Every `\TODO{...}` in the LaTeX text has a key here.

In [17]:
def pct(x):
    return f"{100 * x:.0f}%"

values = {
    "bot_comment_share": pct(c25.is_bot.mean()),
    "prs_api": int(items.is_pr.sum()), "issues_api": int((~items.is_pr).sum()),
    "human_comments": int(monthly.loc["total", "human_comments"]),
    "review_comments": int(monthly.loc["total", "review_comments"]),
    "commenters": int(monthly.loc["total", "commenters"]),
    "lgtm_half": lgtm_half, "lgtm_people": lgtm_people,
    "approve_half": approve_half, "approve_people": approve_people,
    "acc_frequent": pct(outcome.loc["Frequent", "acceptance"]),
    "acc_newcomer": pct(outcome.loc["Newcomer", "acceptance"]),
    "ttfr_pr_frequent": ttfr_pr.loc["Frequent", "median_days"],
    "ttfr_pr_newcomer": ttfr_pr.loc["Newcomer", "median_days"],
    "noresp_pr_newcomer": pct(ttfr_pr.loc["Newcomer", "no_human_response"]),
    "noresp_pr_frequent": pct(ttfr_pr.loc["Frequent", "no_human_response"]),
    "ttfr_issue_newcomer": ttfr_issue.loc["Newcomer", "median_days"],
    "stale_newcomer": pct(bots_table.loc["Newcomer", "marked stale before any human response"]),
    "stale_frequent": pct(bots_table.loc["Frequent", "marked stale before any human response"]),
    "botclose_newcomer": pct(bots_table.loc["Newcomer", "of which closed by the bot"]),
    "gated_newcomer": pct(ok_table.loc["Newcomer", "share of PRs gated"]),
    "okwait_newcomer": ok_table.loc["Newcomer", "median wait (days)"],
    "ttfr_issue_frequent": ttfr_issue.loc["Frequent", "median_days"],
    "noresp_issue_newcomer": pct(ttfr_issue.loc["Newcomer", "no_human_response"]),
    "approvers_reviewing": f"{len(approvers & active_reviewers)} of {len(approvers)} ({pct(len(approvers & active_reviewers) / len(approvers))})",
}
pd.Series(values, name="value").to_frame()

,value
bot_comment_share,27%
prs_api,4567
issues_api,1859
human_comments,47278
review_comments,29183
commenters,2515
lgtm_half,11
lgtm_people,176
approve_half,4
approve_people,95
